<a href="https://www.kaggle.com/code/dulapurkaystha/snack-guardian-ai?scriptVersionId=281768064" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Snack Guardian AI: A Multi-Agent Gut-Friendly Snack Assistant

In [13]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [14]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService

print("✅ ADK components imported successfully.")

retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

model = Gemini(
    model_name="gemini-2.5-flash-lite",
    retry_options=retry_config
)

APP_NAME = "default"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session

✅ ADK components imported successfully.


In [15]:
root_agent = Agent(
    name="helpful_snack_agent",
    model=model,
    # description="A simple snack agent that can suggest snacks."
    instruction="""
    You are a helpful assistant.
        
    - When the user tells you about a new trigger, safe food, preference,
      or something they never want suggested again, call
      update_user_profile(user_id="default_user", key=..., value=...).

    - Use these keys when appropriate:
        - "known_triggers" for foods that cause problems (e.g. popcorn)
        - "safe_foods" for foods that feel good
        - "likes" for textures/flavors they enjoy (e.g. warm, crunchy)
        - "dislikes" for things they don't like
        - "never_suggest" for things never to recommend

    - When the user asks what you know about them, call
      get_user_profile(user_id="default_user") and summarize.

    Always use these tools instead of guessing.
    """,
    tools=[google_search],
)

db_url = "sqlite:///my_profile_data.db"
session_service = DatabaseSessionService(db_url=db_url)

print(f"   - Database: my_agent_data.db")

print("✅ Root Agent defined.")

   - Database: my_agent_data.db
✅ Root Agent defined.


In [16]:
runner = Runner(
    agent=root_agent, 
    app_name=APP_NAME, 
    session_service=session_service,
)

print("✅ Runner created.")

✅ Runner created.


In [17]:
# response = await runner.run_debug(
#     "I have mild acid reflux. Can you suggest a gentle evening snack?"
# )

response = await runner.run_debug(
    "My name is Sally. Remember that I cannot eat spicy foods."
)

response = await runner.run_debug(
    "What do you know about my gut triggers now? and What's my name",
)

print(get_user_profile("default_user"))


 ### Created new session: debug_session_id

User > My name is Sally. Remember that I cannot eat spicy foods.
helpful_snack_agent > I've noted that you cannot eat spicy foods, Sally. I'll remember that for future suggestions.

 ### Continue session: debug_session_id

User > What do you know about my gut triggers now? and What's my name
helpful_snack_agent > I'm sorry, Sally, but I'm currently experiencing a technical issue and am unable to access or update your profile. Therefore, I can't tell you what I know about your gut triggers at this moment.

However, I do remember that your name is Sally, and that you mentioned you cannot eat spicy foods. I will remember this in our conversation.


NameError: name 'get_user_profile' is not defined